# Clase 205 — SHAP / LIME / PDP / ICE en producción

TreeSHAP vs KernelSHAP, PDP+ICE, comparación con LIME, y diseño de un endpoint `/explain`.

Requiere: `pip install shap lime xgboost matplotlib`.

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import xgboost as xgb

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1).fit(Xtr, ytr)
print('test R2:', model.score(Xte, yte))

## 1. TreeSHAP — rápido y exacto

In [ ]:
import shap
expl = shap.TreeExplainer(model)
t0 = time.perf_counter()
shap_vals = expl(Xte.iloc[:500])
print(f'TreeSHAP 500 instancias: {time.perf_counter() - t0:.3f} s')

# Sanity: SHAP values + base = predicción
i = 0
recon = expl.expected_value + shap_vals.values[i].sum()
pred = model.predict(Xte.iloc[[i]])[0]
print(f'instance {i}: SHAP-reconstructed={recon:.4f} ; model.predict={pred:.4f}')

In [ ]:
# Top features para una instancia (lo que devolvería /explain)
def top_k_explanation(shap_one, feature_names, k=5):
    pairs = sorted(zip(feature_names, shap_one.values), key=lambda p: abs(p[1]), reverse=True)
    return [{'name': n, 'shap': float(v)} for n, v in pairs[:k]]

explanation = {
    'prediction': float(model.predict(Xte.iloc[[0]])[0]),
    'base_value': float(expl.expected_value),
    'top_features': top_k_explanation(shap_vals[0], X.columns),
}
import json; print(json.dumps(explanation, indent=2))

## 2. KernelSHAP — agnóstico (lento)

In [ ]:
background = shap.sample(Xtr, 50, random_state=0)
kexpl = shap.KernelExplainer(model.predict, background)
t0 = time.perf_counter()
kvals = kexpl.shap_values(Xte.iloc[:10], nsamples=100, silent=True)
print(f'KernelSHAP 10 instancias × 100 samples: {time.perf_counter() - t0:.2f} s')

# Correlación TreeSHAP vs KernelSHAP en las primeras 10 instancias
tree_first10 = shap_vals.values[:10]
corr = np.corrcoef(tree_first10.ravel(), kvals.ravel())[0, 1]
print(f'correlación TreeSHAP vs KernelSHAP: {corr:.3f}  (alto → consistentes)')

## 3. PDP + ICE

In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
PartialDependenceDisplay.from_estimator(
    model, Xtr.sample(500, random_state=0),
    features=['MedInc', 'HouseAge'],
    kind='both',  # 'average' = PDP, 'individual' = ICE, 'both' = ambos
    ax=ax, ice_lines_kw={'alpha': 0.3, 'color': 'gray'},
    pd_line_kw={'color': 'red', 'linewidth': 3},
)
plt.tight_layout()
plt.savefig('pdp_ice.png', dpi=100)
plt.show()
print('PDP en rojo, ICE en gris. Heterogeneidad = ICEs no paralelos.')

## 4. LIME — comparación local

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
lexpl = LimeTabularExplainer(
    training_data=Xtr.values, feature_names=list(X.columns),
    mode='regression', random_state=42,
)
i = 0
lime_exp = lexpl.explain_instance(Xte.values[i], model.predict, num_features=5)
print(f'LIME top-5 para instancia {i}:')
for f, w in lime_exp.as_list():
    print(f'  {f}: {w:+.4f}')
print(f'\nSHAP top-5 para instancia {i}:')
for x in top_k_explanation(shap_vals[i], X.columns):
    print(f'  {x["name"]}: {x["shap"]:+.4f}')

## 5. Endpoint `/explain` (stub FastAPI)

In [ ]:
endpoint = '''\
from fastapi import FastAPI
from pydantic import BaseModel
import shap, xgboost as xgb, joblib, numpy as np

MODEL = joblib.load("model.pkl")
EXPL = shap.TreeExplainer(MODEL)        # init UNA vez
FEATURE_NAMES = [...]                    # cargar al startup

app = FastAPI()

class ExplainIn(BaseModel):
    features: list[float]
    top_k: int = 5

@app.post("/explain")
def explain(x: ExplainIn):
    arr = np.asarray(x.features).reshape(1, -1)
    pred = float(MODEL.predict(arr)[0])
    sv = EXPL(arr)
    pairs = sorted(zip(FEATURE_NAMES, sv.values[0]), key=lambda p: abs(p[1]), reverse=True)
    return {
        "prediction": pred,
        "base_value": float(EXPL.expected_value),
        "top_features": [{"name": n, "shap": float(v)} for n, v in pairs[:x.top_k]],
    }
'''
print(endpoint)

## Ejercicio guiado

1. Cachéa `shap.TreeExplainer(model)` al startup y medí latencia de `/explain` — debería ser <50 ms p99 para 1 instancia.
2. Calculá la global summary `shap.plots.bar(shap_vals)` y persistila como JSON para servir desde `GET /global-explanation`.
3. Comparé PDP vs ALE (`pip install PyALE`) sobre features correladas (`MedInc` vs `AveRooms`). Mostrá un caso donde PDP miente y ALE acierta.
4. Para una instancia donde SHAP y LIME difieren, investigá por qué.
5. Bonus: monitor de explicaciones — si la distribución de feature importance cambia mucho entre dos snapshots semanales, hay drift conceptual.

## Conclusiones

- TreeSHAP es 100-1000× más rápido que KernelSHAP — usalo siempre que el modelo sea árbol.
- PDP es popular pero engañoso con features correladas; ALE es la versión correcta.
- `/explain` se construye cacheando el explainer al startup, no creándolo por request.
- Comunicación a stakeholders ≠ output crudo de SHAP — traducir nombres y mostrar dirección.